<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-1-prompt-engineering/Session_2_Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 — Prompt Engineering in Code
## Session 2 — Prompt Chaining

In this session we build a two-step prompt chain.
Step 1 — Extract key details from a Swiggy customer complaint
Step 2 — Use those details to generate a professional support reply

We first build it in raw Python, then refactor using LangChain.

In [1]:
# Install required libraries
!pip install groq langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00


In [2]:
## Setup — Load API Key and Model
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Load API key safely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Create the model object
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Model ready!")

Model ready!


## Step 1 — Extract Order Details from Complaint (Raw Python)
We send the complaint to the AI and extract structured details.

In [6]:
def extract_complaint_details(complaint):
    prompt = f"""You are a Swiggy customer support assistant.
Extract the following details from this customer complaint:
- Issue type (one of: DELIVERY, FOOD_QUALITY, PAYMENT, APP_ISSUE, OTHER)
- Sentiment (one of: ANGRY, NEUTRAL, POLITE)
- Key problem in one short sentence

Reply in exactly this format:
Issue: <issue type>
Sentiment: <sentiment>
Problem: <key problem>

Complaint: {complaint}"""

    response = llm.invoke(prompt)
    return response.content

# Test with a Swiggy complaint
complaint = "This is ridiculous! I waited 1 hour for my biryani and it arrived completely cold. I want a refund immediately!"

details = extract_complaint_details(complaint)
print("Extracted Details:")
print(details)

Extracted Details:
Issue: FOOD_QUALITY
Sentiment: ANGRY
Problem: Food arrived cold.


## Step 2 — Generate Support Reply Using Extracted Details
We use the details from Step 1 to craft a professional support reply.

In [8]:
# Step 2 of our chain — Generate a support reply
# Notice how we use the OUTPUT of step 1 as INPUT here

def generate_support_reply(details, original_complaint):
    prompt = f"""You are a professional Swiggy customer support agent.

A customer has sent a complaint. Here are the extracted details:
{details}

Original complaint: {original_complaint}

Write a professional, empathetic support reply that:
- Always greet the customer as "Dear Customer"
- If sentiment is ANGRY, use extra empathetic and apologetic language
- Acknowledges the specific problem mentioned
- Offers a concrete solution (refund or replacement)
- Ends with a positive note
- Always address the customer as "Dear Customer", never use their sentiment as their name

Keep the reply under 5 sentences."""

    response = llm.invoke(prompt)
    return response.content

# Test Step 2 using the output from Step 1
reply = generate_support_reply(details, complaint)
print("Support Reply:")
print(reply)

Support Reply:
Dear Customer,

I am deeply sorry to hear that your biryani arrived cold after waiting for an hour. I can imagine how frustrating and disappointing this experience must be for you. As a gesture of goodwill, I would like to offer you a full refund for your order. Please allow me to assist you with the refund process, and I will ensure that it is processed as soon as possible. Your satisfaction is our top priority, and I appreciate your feedback in helping us improve our services.

Best regards,
Swiggy Customer Support


## Full Chain — Connect Step 1 and Step 2
We combine both steps into one function.
Input: raw complaint
Output: professional support reply

In [9]:
# Full two-step chain in raw Python
# Input: raw complaint text
# Output: professional support reply

def swiggy_support_chain(complaint):

    print("Step 1 — Extracting complaint details...")
    details = extract_complaint_details(complaint)
    print(details)
    print("\nStep 2 — Generating support reply...")
    reply = generate_support_reply(details, complaint)
    print(reply)

    return {
        "complaint": complaint,
        "details": details,
        "reply": reply
    }

# Test the full chain with a new complaint
result = swiggy_support_chain(
    "I paid twice for my order but only received one delivery. This is unacceptable!"
)

Step 1 — Extracting complaint details...
Issue: PAYMENT
Sentiment: ANGRY
Problem: Customer was charged twice for a single delivery.

Step 2 — Generating support reply...
Dear Customer,

I am deeply sorry to hear that you were charged twice for a single delivery, and we fell short of your expectations. I can imagine how frustrating this must be for you, and I want to assure you that we're taking immediate action to rectify the situation. We will process a full refund for the duplicate payment, and you should receive it within the next 24 hours. Please allow us some time to investigate and prevent such incidents in the future. Thank you for your patience and understanding, and we hope to serve you better next time.

Best regards,
Swiggy Customer Support


## Refactoring with LangChain
We rebuild the same chain using LangChain's PromptTemplate and LLMChain.
This is cleaner, more reusable, and industry standard.

In [11]:
# LangChain recently moved some components to separate packages
# PromptTemplate now lives in langchain_core
# LLMChain now lives in langchain — but we use a newer approach instead

from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Step 1 — Extraction prompt as a LangChain PromptTemplate
extraction_prompt = PromptTemplate(
    input_variables=["complaint"],
    template="""You are a Swiggy customer support assistant.
Extract the following details from this customer complaint:
- Issue type (one of: DELIVERY, FOOD_QUALITY, PAYMENT, APP_ISSUE, OTHER)
- Sentiment (one of: ANGRY, NEUTRAL, POLITE)
- Key problem in one short sentence

Reply in exactly this format:
Issue: <issue type>
Sentiment: <sentiment>
Problem: <key problem>

Complaint: {complaint}"""
)

# In modern LangChain we use | operator instead of LLMChain
# This is called LCEL — LangChain Expression Language
# Think of | like a pipe — complaint goes in, details come out
extraction_chain = extraction_prompt | llm

print("Extraction chain ready!")

Extraction chain ready!


In [12]:
# Step 2 — Reply generation prompt as a LangChain PromptTemplate
# Notice it takes TWO variables — details AND original_complaint
reply_prompt = PromptTemplate(
    input_variables=["details", "original_complaint"],
    template="""You are a professional Swiggy customer support agent.

A customer has sent a complaint. Here are the extracted details:
{details}

Original complaint: {original_complaint}

Write a professional, empathetic support reply that:
- Always greet the customer as "Dear Customer"
- If sentiment is ANGRY, use extra empathetic and apologetic language
- Acknowledges the specific problem mentioned
- Offers a concrete solution (refund or replacement)
- Ends with a positive note

Keep the reply under 5 sentences."""
)

# Connect reply prompt to model
reply_chain = reply_prompt | llm

print("Reply chain ready!")

Reply chain ready!


In [13]:
# Full LangChain pipeline
# We connect extraction_chain and reply_chain manually
# Input: raw complaint
# Output: professional support reply

def swiggy_support_chain_langchain(complaint):

    print("Step 1 — Extracting complaint details...")
    # .invoke() runs the chain with the given input
    # we pass a dictionary with the variable name matching input_variables
    details_response = extraction_chain.invoke({"complaint": complaint})
    details = details_response.content
    print(details)

    print("\nStep 2 — Generating support reply...")
    reply_response = reply_chain.invoke({
        "details": details,
        "original_complaint": complaint
    })
    reply = reply_response.content
    print(reply)

    return {
        "complaint": complaint,
        "details": details,
        "reply": reply
    }

# Test with a new Zomato style complaint
result = swiggy_support_chain_langchain(
    "I ordered a Dominos pizza on Swiggy but received a completely different restaurant's food. Very disappointed!"
)

Step 1 — Extracting complaint details...
Issue: DELIVERY
Sentiment: ANGRY
Problem: Received wrong food order.

Step 2 — Generating support reply...
Dear Customer,

I am deeply sorry to hear that you received the wrong food order from us, and it's completely unacceptable that you were served a different restaurant's food instead of your ordered Dominos pizza. I can imagine how frustrating and disappointing this experience must be for you. Please allow me to assist you further - I'd be happy to process a full refund for your order, and we'll also provide a voucher for your next order as a gesture of goodwill. Your satisfaction is our top priority, and I assure you that we're taking immediate action to prevent such incidents in the future.

Best regards, [Your Name]


In [14]:
# Compare raw Python chain vs LangChain chain
import pandas as pd

test_complaints = [
    "My order was 45 minutes late and food was cold",
    "I was charged twice for my order",
    "Wrong items were delivered"
]

results = []

for complaint in test_complaints:
    # Raw Python chain
    raw_result = swiggy_support_chain(complaint)

    # LangChain chain
    lc_result = swiggy_support_chain_langchain(complaint)

    results.append({
        "Complaint": complaint,
        "Raw Python Details": raw_result["details"].split("\n")[0],  # just first line
        "LangChain Details": lc_result["details"].split("\n")[0],    # just first line
    })

df = pd.DataFrame(results)
df

Step 1 — Extracting complaint details...
Issue: DELIVERY
Sentiment: ANGRY
Problem: Order was significantly delayed and food was served cold.

Step 2 — Generating support reply...
Dear Customer,

I am truly sorry to hear that your order was significantly delayed and served cold, which must have been extremely frustrating for you. I can imagine how disappointing it must be to wait for 45 minutes and receive food that's not up to your expectations. As a gesture of goodwill, I'd like to offer you a full refund for your order. Please let me know if there's anything else I can do to make things right. We value your trust and look forward to serving you better in the future.

Best regards,
Swiggy Customer Support
Step 1 — Extracting complaint details...
Issue: DELIVERY
Sentiment: ANGRY
Problem: Order was significantly delayed and food was served cold.

Step 2 — Generating support reply...
Dear Customer,

I am truly sorry to hear that your order was significantly delayed and served cold, which

,Complaint,Raw Python Details,LangChain Details
0,My order was 45 minutes late and food was cold,Issue: DELIVERY,Issue: DELIVERY
1,I was charged twice for my order,Issue: PAYMENT,Issue: PAYMENT
2,Wrong items were delivered,Issue: DELIVERY,Issue: DELIVERY


## Key Takeaways — Session 2

- Prompt chaining means output of one prompt becomes input of the next
- Raw Python chains give full control and show fundamentals
- LangChain PromptTemplate makes prompts reusable and clean
- The | operator connects prompt to model in modern LangChain
- Both approaches produce same quality results — LangChain just organises better